In [ ]:
# Imports
import sys
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt

In [ ]:
# CD-NLGSSM imports
from cd_dynamax import make_key_sequence
from cd_dynamax.src.utils.experiment_utils import *
sys.path.append("..")
from data_generator import create_cdnlgssm_model_from_config

## Define the Lorenz 63 Model


We generate data from a Lorenz 63 system, from dynamics with the following stochastic differential equations:

\begin{align*}
\frac{d x}{d t} &= a(y-x) + \sigma w_x(t) \\
\frac{d y}{d t} &= x(b-z) - y + \sigma w_y(t) \\
\frac{d z}{d t} &= xy - cz + \sigma w_z(t),
\end{align*}

With parameters $a=10, b=28, c=8/3$, the system gives rise to chaotic behavior, and we choose $\sigma=1.0$ for diffusion.

To generate data, we numerically approximate random path solutions to this SDE using Heun's method (i.e. improved Euler), as implemented in [Diffrax](https://docs.kidger.site/diffrax/api/solvers/sde_solvers/).


We assume the observation model is
\begin{align*}
y(t) &= H x(t) + r(t) \\
r(t) &\sim N(0,R),
\end{align*}
where we choose $R=I$. 

Namely, **we impose partial observability with H=[1, 0, 0]**, with noisy observations, sampled at irregular time intervals.

In [ ]:
# Default Lorenz 63 parameter definition in config file
# where only the first component (x_1) is observed
default_lorenz63_config_model = "../configs/model/l63_x1_likelihood"

In [ ]:
# Specify true Lorenz '63 parameters
from cd_dynamax.src.utils.physics_based_models import LearnableLorenz63_Drift

# Define the True Lorenz 63 Model parameters dictionary
true_model_def = {
    'initial_values.dynamics_drift': {
        "params": LearnableLorenz63_Drift(
            sigma=10.0,
            rho=28.0,
            beta=8.0 / 3.0
        ),
        "props": None # Let us use the default parameter properties
    }
}


In [ ]:
# Create and initialize the CD-NLGSSM model
true_model, true_params, true_props = create_cdnlgssm_model_from_config(
    true_model_config_file=default_lorenz63_config_model,
    overrides=true_model_def,
)

### Simulate data using true CDNLGSSM model

In [ ]:
# Simulate irregular emission times
keys = make_key_sequence(0)
t_emissions = sample_t_emissions(start=0.0, stop=100.0, dt=0.1, regular=False, key=next(keys))

# Simulate data using true parameters (observed at times t_emissions)
# Note that default SDE diffeqsolve settings solve using dfx.Heun() with dfx.ConstantStepSize() and dt0=0.01 step size.
states, emissions = true_model.sample(
    params=true_params,
    key=next(keys),
    num_timesteps=len(t_emissions),
    t_emissions=t_emissions,
    transition_type="path",
)

In [ ]:
# Plot the results in a subplot with 3 rows, with observations overlaid in first emission_dims rows
fig, axs = plt.subplots(
    true_model.state_dim,
    1,
    figsize=(10, 8),
    sharex=True
)
state_labels = ['x', 'y', 'z']

for i in range(true_model.state_dim):
    axs[i].plot(
        t_emissions, states[:, i],
        label=f'State {state_labels[i]}',
        color='C0'
    )
    if i < true_model.emission_dim:
        axs[i].scatter(
            t_emissions,
            emissions[:, i],
            label=f'Observations of {state_labels[i]}',
            color='C1',
            s=10
        )
    axs[i].legend()
    axs[i].set_ylabel(f'State {state_labels[i]}')
axs[-1].set_xlabel('Time')
plt.suptitle('Lorenz \'63 System States and Observations')
plt.tight_layout()
plt.show()

Now perform filtering over the observed emissions using EnKF.
This produces filtered state estimates at the observation times, but also, crucially, computes an estimate of the log likelihood of the observed data under the model.
We define the function below, and run it with different models

In [ ]:
def filter_and_compute_loglik(model, params, title="True parameters", T=None):
    
    if T is None:
        T = len(emissions)

    emissions_to_use = emissions[:T]
    t_emissions_to_use = t_emissions[:T]
    states_to_use = states[:T]
    

    # Note that default SDE diffeqsolve settings solve using dfx.Heun() with dfx.ConstantStepSize() and dt0=0.01 step size.
    filtered = model.filter(
        params=params,
        emissions=emissions_to_use,
        t_emissions=t_emissions_to_use,
        key=next(keys),
        enkf_N_particles=100,
    )

    print(f"Log likelihood {title}:", float(filtered.marginal_loglik))

    # Plot a subfigure with state_dim rows showing true state, filtered mean and (optionally via shaded region) filtered stddev.
    # Also plot the observations in the first emission_dim rows.
    fig, axs = plt.subplots(model.state_dim, 1, figsize=(10, 8), sharex=True)
    for i in range(model.state_dim):
        axs[i].plot(t_emissions_to_use, states_to_use[:, i], label=f'True State {state_labels[i]}', color='C0')
        axs[i].plot(t_emissions_to_use, filtered.filtered_means[:, i], label=f'Filtered Mean {state_labels[i]}', color='C1')
        axs[i].fill_between(
            t_emissions_to_use.squeeze(),
            filtered.filtered_means[:, i] - 2 * jnp.sqrt(filtered.filtered_covariances[:, i, i]),
            filtered.filtered_means[:, i] + 2 * jnp.sqrt(filtered.filtered_covariances[:, i, i]),
            color='C1',
            alpha=0.3,
            label=f'Filtered ±2 Stddev {state_labels[i]}'
        )
        if i < model.emission_dim:
            axs[i].scatter(t_emissions_to_use, emissions_to_use[:, i], label=f'Observations of {state_labels[i]}', color='C2', s=10)
        axs[i].legend(loc='upper right')
        axs[i].set_ylabel(f'State {state_labels[i]}')
    axs[-1].set_xlabel('Time')
    plt.suptitle(title)
    plt.tight_layout()

    return filtered.marginal_loglik
        

In [ ]:
# Create two new models, with different drift functions
slightly_bad_drift = {
    'initial_values.dynamics_drift': {
        "params": LearnableLorenz63_Drift(
            sigma=11.0,
            rho=28.0,
            beta=8.0 / 3.0
        ),
        "props": None # Let us use the default parameter properties
    }
}

# Create and initialize the CD-NLGSSM model
slightly_bad_model, slightly_bad_params, _ = create_cdnlgssm_model_from_config(
    true_model_config_file=default_lorenz63_config_model,
    overrides=slightly_bad_drift,
)

# Very Mis-Specified model
very_bad_drift = {
    'initial_values.dynamics_drift': {
        "params": LearnableLorenz63_Drift(
            sigma=25.0,
            rho=15.0,
            beta=10.0
        ),
        "props": None # Let us use the default parameter properties
    }
}

# Create and initialize the CD-NLGSSM model
very_bad_model, very_bad_params, _ = create_cdnlgssm_model_from_config(
    true_model_config_file=default_lorenz63_config_model,
    overrides=very_bad_drift,
)


Now we can compute the log likelihood at the true parameters, then at the mis-specified parameters. We will visualize the resulting filtering distributions as well to give intuitition for why the log likelihoods differ. 

The likelihood should decrease when the parameters get further from the truth---the true parameters should yield the highest log-likelihood. If they don't, then the finite-amount of noisy data is at likely play---those two models are not distinguishable given the data. This is typically why we use Bayesian methods to quantify uncertainty in parameter estimates.

CAVEAT: the likelihood approximation is stochastic and approximate.

In [ ]:
# Run for 10% of data for easy visualization and fast computation.
filter_and_compute_loglik(
    true_model,
    true_params,
    title="Filtering Results at True Parameters", T=len(emissions)//10
)
filter_and_compute_loglik(
    slightly_bad_model,
    slightly_bad_params,
    title="Filtering Results at Slightly Bad Parameters",
    T=len(emissions)//10
)
filter_and_compute_loglik(
    very_bad_model,
    very_bad_params,
    title="Filtering Results at Very Bad Parameters",
    T=len(emissions)//10
)

Now, let's sweep over Rho (holding sigma and beta fixed at true values) to see how the log likelihood varies with this parameter. We should see a peak around the true value of Rho=28.0, but we know that the amount of data is limited and the likelihood is approximate, so we may not see this exactly. If you want to see an unbiased peak, try increasing the amount of data (T) and number of particles.


In [ ]:
def compute_likelihood_of_model(model, params, T=None, N_particles=100):
        
    if T is None:
        T = len(emissions)
    
    emissions_to_use = emissions[:T]
    t_emissions_to_use = t_emissions[:T]

    # Run filtering to compute log likelihood
    filtered = model.filter(
        params=params,
        emissions=emissions_to_use,
        t_emissions=t_emissions_to_use,
        key=next(keys), # if you comment this out, you'll get deterministic results (the EnKF will always be seeded with jr.PRNGKey(0))
        enkf_N_particles=N_particles,
        warn=False,
    )
    
    return filtered.marginal_loglik

In [ ]:
def rho_sweep_experiment(true_rho=28.0,T=None, N_particles=100, do_slow_loops=False):
    if T is None:
        T = len(emissions)

    def _compute_likelihood_of_model(model, params):
        return compute_likelihood_of_model(model, params, T=T, N_particles=N_particles)
    
    print("Building model and batched parameters...")
    all_params = []
    model = None  # We only need to create the model object once
    rho_values = jnp.linspace(10.0, 50.0, 20)
    for rho in rho_values:
        # Create model and params for this rho
        current_model, current_params, _ = create_cdnlgssm_model_from_config(
            true_model_config_file=default_lorenz63_config_model,
            overrides={
                'initial_values.dynamics_drift': {
                    "params": LearnableLorenz63_Drift(
                        sigma=10.0,
                        rho=float(rho), # Pass a standard float
                        beta=8.0/3.0
                    ),
                    "props": None
                },
            }
        )
        
        if model is None:
            model = current_model  # Store the model (assumed to be static)
        
        all_params.append(current_params)

    # This stacks all parameter leaves into a single PyTree
    # where each leaf has a new leading (batch) dimension.
    batched_params = jax.tree_util.tree_map(lambda *x: jnp.stack(x), *all_params)

    # In JAX, we can use vmap to compute log likelihoods in parallel (via vectorization). Let's time it, and compare to a for-loop and a lax.scan (a JAX sped-up version of a for-loop).
    from time import time
    # Using vmap
    from jax import vmap
    start_time = time()
    print("Note that vmap can trigger un-hit warnings, so don't be alarmed if you see them here.")
    loglik_values_vmap = vmap(
        _compute_likelihood_of_model,
        in_axes=(None,0)
        )(model, batched_params)
    vmap_time = time() - start_time
    print(f"vmap time: {vmap_time:.2f} seconds")

    if do_slow_loops:
        # Using for-loop
        start_time = time()
        loglik_values_loop = []
        for param in all_params:
            loglik = _compute_likelihood_of_model(model, param)
            loglik_values_loop.append(loglik)
        loglik_values_loop = jnp.array(loglik_values_loop)
        loop_time = time() - start_time
        print(f"For-loop time: {loop_time:.2f} seconds")

        # Using lax.scan
        from jax import lax
        def scan_step(carry, param):
            loglik = _compute_likelihood_of_model(model, param)
            return carry, loglik
        start_time = time()
        _, loglik_values_scan = lax.scan(scan_step, None, batched_params)
        scan_time = time() - start_time
        print(f"lax.scan time: {scan_time:.2f} seconds")

        # Using pmap if you have access to multiple devices (this is parallelization across devices, not vectorization)
        try:
            from jax import pmap
            start_time = time()
            loglik_values_pmap = pmap(
                _compute_likelihood_of_model,
                in_axes=(None,0)
            )(model, batched_params)
            pmap_time = time() - start_time
            print(f"pmap time: {pmap_time:.2f} seconds")
        except Exception as e:
            print("pmap not available or failed to run:", e)
    
    # Plot the log likelihood vs Rho
    # Note that because the likelihood is stochastic and approximate, the curves may not be perfectly smooth or peak exactly at the true parameter value.
    # These curves should be similar, but may have slight differences due to the stochastic nature of the EnKF at each call.
    # That is, there is no difference between vmap, for-loop, and lax.scan in terms of the results other than the order of random numbers used (and POSSIBLY some small hardware-dependent numerical differences).
    # To get perfectly identical results, you would need to fix the random seed inside compute_likelihood_at_sigma, which is shown above in the comments.
    plt.figure(figsize=(10, 6))
    plt.plot(rho_values, loglik_values_vmap, label='vmap', marker='o')
    if do_slow_loops:
        plt.plot(rho_values, loglik_values_loop, label='for-loop', marker='x')
        plt.plot(rho_values, loglik_values_scan, label='lax.scan', marker='s')
    plt.axvline(x=true_rho, color='k', linestyle=':', label='True Rho')
    plt.xlabel('Rho Value')
    plt.ylabel('Log Likelihood')
    plt.title(f'Log Likelihood vs Rho Value for data up to T={T} with N_particles={N_particles}')
    plt.legend()
    plt.show()

In [ ]:
# Run sweep experiment with all data
rho_sweep_experiment(
    T=len(emissions),
    N_particles=100
)

In [ ]:
# Run sweep experiment with all data but few particles (bad approximation). Notice that it is jagged, and does not peak at the true value. This is due to high variance in the likelihood estimate with few particles.
rho_sweep_experiment(
    T=len(emissions),
    N_particles=10
)

In [ ]:
# Run sweep experiment with 10% of data. "Do slow loops" to see timing comparison between vmap, for-loop, and lax.scan.
# Here, we see that the likelihood peaks near the true value, but it is very flat in that region---that is due to limited data. 100 particles is likely sufficient to get a good approximation here.
rho_sweep_experiment(
    T=len(emissions)//10,
    N_particles=100,
    do_slow_loops=True
)

In [ ]:
# Run sweep experiment with 10% of data and few particles.
# Now with small data and few particles, the likelihood is very noisy and also does not peak exactly at the true value (due to both limited data and high variance in the likelihood estimate).
# This is the noisiest of the experiments, and you can imagine that using such a likelihood could make parameter estimation / uncertainty quantification quite difficult! e.g., it introduces a spurious multimodality.
rho_sweep_experiment(
    T=len(emissions)//10,
    N_particles=10,
    do_slow_loops=True
)